<a href="https://colab.research.google.com/github/Kauustubbh/DocForgery/blob/main/notebooks/02_model_training_baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Cell 1: Cloud Environment Setup & Data Scratchpad
Mounts Google Drive, reads your Kaggle secret, downloads the raw CASIA dataset to local /content/data/, and pulls the CASIA 2.0 ground-truth masks.

In [1]:
!pip install -q segmentation-models-pytorch

In [2]:
# Cell 1: Environment Setup, Drive Mount & Data Scratchpad
import os
from google.colab import drive, userdata

# 1. Mount Google Drive and define persistent root
drive.mount('/content/drive')
PROJECT_DIR = '/content/drive/MyDrive/DocForgery'
CHECKPOINTS_DIR = os.path.join(PROJECT_DIR, 'checkpoints')
os.makedirs(CHECKPOINTS_DIR, exist_ok=True)

# 2. Authenticate Kaggle CLI using Colab Secret
try:
    token = userdata.get('Kaggle_api_key')
except:
    token = userdata.get('KAGGLE_API_TOKEN')

os.environ['KAGGLE_API_TOKEN'] = token

# 3. Download & extract raw images and ground-truth masks into local scratchpad
if not os.path.exists('data/CASIA2'):
    print("Downloading CASIA 2.0 images...")
    !kaggle datasets download -d sophatvathana/casia-dataset --unzip -p data/

if not os.path.exists('data/CASIA2_Groundtruth'):
    print("Downloading CASIA 2.0 ground-truth masks...")
    !git clone --depth 1 https://github.com/namtpham/casia2groundtruth.git data/casia2_gt
    !unzip -q data/casia2_gt/CASIA2.0_Groundtruth.zip -d data/CASIA2_Groundtruth/

print("✅ Environment ready and dataset scratchpad restored.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Environment ready and dataset scratchpad restored.


## Cell 2: Ingest Split Contract (metadata.csv)
Loads the exact metadata.csv manifest generated and balanced in Notebook 01 from Google Drive, verifies local file paths, and isolates the training and validation partitions.

In [3]:
# Load Split Contract from Drive
import os
import pandas as pd

# 1. Read manifest contract exported by Notebook 01 from Google Drive
manifest_path = os.path.join(PROJECT_DIR, 'processed', 'metadata.csv')
df = pd.read_csv(manifest_path)

# 2. Verify local file availability
assert os.path.exists(df.iloc[0]['image_path']), f"Missing image file at: {df.iloc[0]['image_path']}"

# 3. Filter train and validation subsets
train_df = df[df['split'] == 'train'].reset_index(drop=True)
val_df = df[df['split'] == 'val'].reset_index(drop=True)

print(f"✅ Split contract loaded successfully from: {manifest_path}")
print(f"   - Training samples:   {len(train_df)} (Authentic: {(train_df['label'] == 0).sum()}, Tampered: {(train_df['label'] == 1).sum()})")
print(f"   - Validation samples: {len(val_df)} (Authentic: {(val_df['label'] == 0).sum()}, Tampered: {(val_df['label'] == 1).sum()})")

✅ Split contract loaded successfully from: /content/drive/MyDrive/DocForgery/processed/metadata.csv
   - Training samples:   2100 (Authentic: 1050, Tampered: 1050)
   - Validation samples: 450 (Authentic: 225, Tampered: 225)


## Cell 3: PyTorch Dataset, Dual Transforms & Fast DataLoaders
Defines synchronized image-and-mask transforms, constructs the custom DocForgeryDataset, configures multi-threaded PyTorch DataLoaders, and sets up the composite BCE + Soft-Dice loss engine.

In [4]:
# Cell 3: Dataset Definition, Dual Augmentation & Loss Function
import random
import cv2
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import torchvision.transforms.functional as TF

class DualTransform:
    """Synchronized spatial transforms for image and binary mask."""
    def __init__(self, size=(256, 256), is_train=True):
        self.size = size
        self.is_train = is_train

    def __call__(self, img: Image.Image, mask: Image.Image):
        # Resize image with bilinear, mask with nearest neighbor
        img = TF.resize(img, self.size, interpolation=TF.InterpolationMode.BILINEAR)
        mask = TF.resize(mask, self.size, interpolation=TF.InterpolationMode.NEAREST)

        if self.is_train:
            # Synchronized Random Horizontal Flip
            if random.random() > 0.5:
                img = TF.hflip(img)
                mask = TF.hflip(mask)
            # Synchronized Random Vertical Flip
            if random.random() > 0.5:
                img = TF.vflip(img)
                mask = TF.vflip(mask)

        # Convert image to tensor and apply standard ImageNet normalization
        img_tensor = TF.to_tensor(img)
        img_tensor = TF.normalize(img_tensor, mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

        # Convert mask to single-channel binary float tensor: shape [1, H, W]
        mask_np = np.array(mask, dtype=np.float32)
        if mask_np.ndim == 3:
            mask_np = mask_np[:, :, 0]
        mask_tensor = torch.from_numpy((mask_np > 128).astype(np.float32)).unsqueeze(0)

        return img_tensor, mask_tensor

class DocForgeryDataset(Dataset):
    def __init__(self, dataframe: pd.DataFrame, transform: DualTransform):
        self.df = dataframe
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row['image_path']).convert('RGB')

        # Load ground truth mask or generate blank mask for authentic documents
        if pd.notnull(row['mask_path']) and os.path.exists(str(row['mask_path'])):
            mask = Image.open(row['mask_path']).convert('L')
        else:
            w, h = image.size
            mask = Image.new('L', (w, h), 0)

        img_tensor, mask_tensor = self.transform(image, mask)
        label = torch.tensor(row['label'], dtype=torch.float32)

        return img_tensor, mask_tensor, label

BATCH_SIZE = 16  # Optimal batch size for 256x256 segmentation on T4 GPU

train_dataset = DocForgeryDataset(train_df, DualTransform(size=(256, 256), is_train=True))
val_dataset = DocForgeryDataset(val_df, DualTransform(size=(256, 256), is_train=False))

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

class CombinedBCEDiceLoss(nn.Module):
    """Composite loss addressing extreme class imbalance in document forgery."""
    def __init__(self, bce_weight=0.5, smooth=1e-6):
        super().__init__()
        self.bce_weight = bce_weight
        self.bce = nn.BCEWithLogitsLoss()
        self.smooth = smooth

    def forward(self, logits, targets):
        bce_loss = self.bce(logits, targets)

        # Soft Dice computation using sigmoid probabilities
        probs = torch.sigmoid(logits)
        intersection = (probs * targets).sum(dim=(2, 3))
        union = probs.sum(dim=(2, 3)) + targets.sum(dim=(2, 3))
        dice_score = (2.0 * intersection + self.smooth) / (union + self.smooth)
        dice_loss = 1.0 - dice_score.mean()

        return self.bce_weight * bce_loss + (1.0 - self.bce_weight) * dice_loss

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
criterion = CombinedBCEDiceLoss().to(device)

print(f"✅ Fast DataLoaders & CombinedBCEDiceLoss initialized on: {device}")

✅ Fast DataLoaders & CombinedBCEDiceLoss initialized on: cuda


## Cell 4: U-Net Architecture, Training Loop & Checkpoint Export
Instantiates a complete, self-contained U-Net localization architecture, trains it across 15 epochs (~12 minutes on T4 GPU), evaluates validation Dice scores every epoch, and writes the best model checkpoint (best_unet.pth) directly to your mounted Google Drive folder.

In [5]:
# Cell 4: Pretrained U-Net (ResNet-34 Backbone), Weighted Loss & Training Loop
import os
import torch
import torch.nn as nn
import torch.optim as optim
import segmentation_models_pytorch as smp
import matplotlib.pyplot as plt

# 1. Instantiate U-Net with ImageNet Pretrained ResNet-34 Encoder
model = smp.Unet(
    encoder_name="resnet34",        # Pretrained backbone
    encoder_weights="imagenet",     # Transfer learning weights
    in_channels=3,
    classes=1
).to(device)

# 2. Weighted Composite Loss (Penalizes missing small forged regions by 10x)
class WeightedBCEDiceLoss(nn.Module):
    def __init__(self, pos_weight=10.0, smooth=1e-6):
        super().__init__()
        self.pos_weight = torch.tensor([pos_weight]).to(device)
        self.bce = nn.BCEWithLogitsLoss(pos_weight=self.pos_weight)
        self.smooth = smooth

    def forward(self, logits, targets):
        bce_loss = self.bce(logits, targets)

        # Soft Dice calculation
        probs = torch.sigmoid(logits)
        intersection = (probs * targets).sum(dim=(2, 3))
        union = probs.sum(dim=(2, 3)) + targets.sum(dim=(2, 3))
        dice_score = (2.0 * intersection + self.smooth) / (union + self.smooth)
        dice_loss = 1.0 - dice_score.mean()

        return 0.4 * bce_loss + 0.6 * dice_loss

criterion = WeightedBCEDiceLoss(pos_weight=10.0).to(device)

# 3. Optimizer with lower learning rate suitable for fine-tuning
optimizer = optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)

NUM_EPOCHS = 15
best_val_loss = float('inf')
checkpoint_path = os.path.join(CHECKPOINTS_DIR, 'best_unet.pth')

history = {'train_loss': [], 'val_loss': [], 'val_dice_tampered': []}

print(f"🚀 Launching Pretrained ResNet-34 U-Net ({NUM_EPOCHS} Epochs on {device})...\n")

# 4. Training Loop
for epoch in range(NUM_EPOCHS):
    model.train()
    running_loss = 0.0
    for images, masks, _ in train_loader:
        images, masks = images.to(device), masks.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, masks)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)

    train_loss = running_loss / len(train_loader.dataset)

    # Validation Phase
    model.eval()
    val_loss = 0.0
    tampered_dice_total = 0.0
    tampered_count = 0

    with torch.no_grad():
        for images, masks, labels in val_loader:
            images, masks, labels = images.to(device), masks.to(device), labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, masks)
            val_loss += loss.item() * images.size(0)

            # Compute Dice exclusively on tampered samples (label == 1)
            probs = torch.sigmoid(outputs)
            preds = (probs > 0.5).float()

            for i in range(len(labels)):
                if labels[i] == 1.0:
                    inter = (preds[i] * masks[i]).sum().item()
                    union = preds[i].sum().item() + masks[i].sum().item()
                    d = (2.0 * inter + 1e-6) / (union + 1e-6)
                    tampered_dice_total += d
                    tampered_count += 1

    val_loss = val_loss / len(val_loader.dataset)
    val_dice_tampered = (tampered_dice_total / tampered_count) if tampered_count > 0 else 0.0
    scheduler.step(val_loss)

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_dice_tampered'].append(val_dice_tampered)

    print(f"Epoch [{epoch+1:02d}/{NUM_EPOCHS:02d}] - Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Tampered Dice: {val_dice_tampered:.4f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), checkpoint_path)
        print(f"   ⭐ Best model checkpoint saved to: {checkpoint_path}")

print(f"\n🎉 Training complete! Checkpoint saved at: {checkpoint_path}")

config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 87.3MB            

model.safetensors: downloading bytes:           |  0.00B            

🚀 Launching Pretrained ResNet-34 U-Net (15 Epochs on cuda)...

Epoch [01/15] - Train Loss: 0.9162 | Val Loss: 0.9004 | Tampered Dice: 0.0882
   ⭐ Best model checkpoint saved to: /content/drive/MyDrive/DocForgery/checkpoints/best_unet.pth
Epoch [02/15] - Train Loss: 0.9024 | Val Loss: 0.9031 | Tampered Dice: 0.1002
Epoch [03/15] - Train Loss: 0.8982 | Val Loss: 0.8976 | Tampered Dice: 0.1289
   ⭐ Best model checkpoint saved to: /content/drive/MyDrive/DocForgery/checkpoints/best_unet.pth
Epoch [04/15] - Train Loss: 0.8891 | Val Loss: 0.8924 | Tampered Dice: 0.1271
   ⭐ Best model checkpoint saved to: /content/drive/MyDrive/DocForgery/checkpoints/best_unet.pth
Epoch [05/15] - Train Loss: 0.8888 | Val Loss: 0.8966 | Tampered Dice: 0.1516
Epoch [06/15] - Train Loss: 0.8862 | Val Loss: 0.8836 | Tampered Dice: 0.1166
   ⭐ Best model checkpoint saved to: /content/drive/MyDrive/DocForgery/checkpoints/best_unet.pth
Epoch [07/15] - Train Loss: 0.8814 | Val Loss: 0.8871 | Tampered Dice: 0.1311
Epo